In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("reading_invalid_data") \
        .getOrCreate()

In [3]:
invalid_data = spark.read.parquet("hdfs:///Data/Silver/Invalid_Looker_Ecommerce_BigQuery_Dataset/order_items")

Add the reason why each row was invalidated

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def broken(cond, message):
    """Return `message` when the rule is broken."""
    return F.when(F.coalesce(cond, F.lit(False)), F.lit(message))

reasons = [
    broken(col("order_item_id").isNull(), "order_item_id is null"),
    broken(col("order_id").isNull(),      "order_id is null"),
    broken(col("user_id").isNull(),       "user_id is null"),
    broken(col("product_id").isNull(),    "product_id is null"),

    broken(col("sale_price").isNull(),    "sale_price is null"),
    broken(col("sale_price") <= 0,        "sale_price <= 0"),
    broken(col("sale_price") >= 100000,   "sale_price >= 100000"),

    broken(col("status_id").isNull(),                     "status_id is null"),
    broken(~col("status_id").isin(0, 1, 2, 3, 4, -1),     "status_id not in allowed values"),

    broken(col("created_at").isNull(),    "created_at is null"),

    broken(col("shipped_at") < col("created_at"),                          "shipped_at is before created_at"),
    broken(col("delivered_at").isNotNull() & col("shipped_at").isNull(),   "delivered_at without shipped_at"),
    broken(col("delivered_at") < col("shipped_at"),                        "delivered_at is before shipped_at"),
    broken(col("returned_at").isNotNull() & col("delivered_at").isNull(),  "returned_at without delivered_at"),
    broken(col("returned_at") < col("delivered_at"),                       "returned_at is before delivered_at"),
]

joined_reasons = F.concat_ws("; ", *reasons)
invalid_data = invalid_data.withColumn(
    "invalid_reason",
    F.when(F.length(joined_reasons) == 0, F.lit("unknown reason")).otherwise(joined_reasons)
)

In [5]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

Show invalid data

In [6]:
invalid_data.show(truncate=False)

+-------------+--------+-------+----------+-----------+--------------------------+--------------------------+--------------------------+-----------+----------+---------+-------------------------------+
|order_item_id|order_id|user_id|product_id|status_name|created_at                |shipped_at                |delivered_at              |returned_at|sale_price|status_id|invalid_reason                 |
+-------------+--------+-------+----------+-----------+--------------------------+--------------------------+--------------------------+-----------+----------+---------+-------------------------------+
|14           |9       |6      |28494     |Shipped    |2022-04-10 02:38:35       |2022-04-08 20:10:00       |null                      |null       |99.83     |1        |shipped_at is before created_at|
|38           |24      |17     |28283     |Complete   |2019-06-18 11:18:10       |2019-06-15 13:55:00       |2019-06-16 12:28:00       |null       |53.97     |2        |shipped_at is before cr

Summary of reasons

In [7]:
(invalid_data
    .select(F.explode(F.split("invalid_reason", "; ")).alias("reason"))
    .groupBy("reason").count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

+-------------------------------+-----+
|reason                         |count|
+-------------------------------+-----+
|shipped_at is before created_at|35872|
+-------------------------------+-----+

